# MODIFICAR
Avance del 23 de octubre
Mario Betancourt

# PARTE 1 - PROPUESTA
En este proyecto se diseñan e implementan algoritmos de aprendizaje por refuerzo profundo para controlar el movimiento de un robot autobalanceante. El robot en cuestión consiste de un cuerpo rígido con dos ruedas acopladas, las cuales son impulsadas por dos motores con cajas reductoras. Para medir su orientación el robot utiliza un sensor IMU (Unidad de Medición Inercial), el cual proporciona información en tiempo real de su vector de aceleración lineal, su vector de velocidad angular y el vector de campo magnético en la posición donde se encuentra ubicado. El algoritmo (físicamente programado en un microcontrolador) usa esta información para controlar el movimiento de los motores y obtener un refuerzo sí esta acción contribuya a que el robot se encuentre balanceado sobre su eje vertical. 

## 1.1. Algoritmos
Se propone comparar y el rendimiento de los cuatro algoritmos vistos durante la sesión de Deep RL parte 2: DDPG, TD3, SAC, PPO. El resultado final de esta comparación debe ser una serie de gráficas que muestren la recompensa en función del número de episodios de entrenamiento. A través de esta gráfica será posible determinar qué algoritmo *aprende más rápido* a cumplir con la tarea y, en estado estacionario, cuál termina completando la misión con el mejor rendimiento.


## 1.2. Fuentes de Incertidumbre
Una de las principales motivaciones para utilizar algoritmos de Aprendizaje por refuerzo para resolver este problema es la cantidad importante de parámetros con incertidumbres asociadas a diferencias de implementación y factores asociados con el ambiente. Se desea que si los algoritmos se implementan sobre un robot físico, estos permitan que, sin importar en qué parte del robot se coloca la IMU, este pueda balancearse a sí mismo.

## 1.3 Antecedentes
Este proyecto está inspirado por la serie de videos de Digikey y Shawn Hymel "Reinforcement Learning for Robotics" donde se implementa un robot autobalanceante con el kit M5Stack BALA2 de Digikey usando un algoritmo PPO. En este proyecto se utilizan las herramientas presentadas en la serie, y se repite con los otros algoritmos vistos en clase, con el objetivo de comparar su rendimiento en esta aplicación. También se implementa una versión física del robot.

# PARTE 2 - ENTORNO

## 2.1. Motor de Físicas MuJoCo
MuJoCo (Multi-Joint dynamics with Contact) es un motor de física de código abierto desarrollado po Google Deepmind. Este está diseñado para realizar simulaciones rápidas y precisas. Sus fortalezas principales son su simulación realista del contacto físico, el uso formatos de modelado simples y compatibles con otros motores de físicas, la inclusión de herramientas de visualización y (posiblemente la más importante), su *fácil integración con IA*. *MuJoCo se puede usar como una librería de Python y se puede integrar directamente con Gymnasium*.

Para utilizar MuJoCo en un programa de Python, es necesario descargarlo

## 2.1. Modelado 3D en CAD
MuJoCo trabaja utilizando mallas de los objetos que componen a un modelo tridimensional del robot. Estos modelos se pueden generar en softwares de diseño asistido por computadora (por sus siglas en inglés, CAD). 

La mayoría de softwares CAD de uso común en ingeniería (E.g. Solidworks, AutoCAD, Autodesk Inventor, Autodesk Fusion, CATIA, entro otros) pueden producir archivos STL (Standard Tessellation Language). El formato STL define la geometría de la superficie de un objeto 3D mediante una malla de triángulos interconectados... ¡Es justo lo que MuJoCo necesita! Por practicidad, se recomienda generar un archivo STL por cada parte con movimiento independiente en el robot. Esto se explicará con más detalle en la próxima sección.

Una observación importante acerca de los modelos de mallas es que estos se vuelven progresivamente más complicados y, por lo tanto, dificiles de utilizar en motores de físicas conforme la cantidad de detalles pequeños incrementa. Esto se debe a que en las regiones con detalles finos la cantidad de triángulos es mayor y por lo tanto se requiere un cómputo más pesado para resolver los contactos. Esto impacta, por ejemplo, en el modelado de las bandas de rodadura de las cubiertas de las ruedas. En su lugar, podemos considerar estas "deformidades" aleatorizando el dominio, como se explicará más adelante.

Respecto a la serie de videos de Shawn Hymel, en este proyecto nos omitimos el proceso de generar un diseño 3D del cuerpo del robot y las ruedas. Utilizamos los archivos STL que utilizan en la serie para importarlos directamente al entorno.

Además de los archivos STL, precisamos de información relevante sobre las propiedades físicas de los objetos que componen el robot. Por cada parte móvil del modelo simplificado, necesitamos su masa, las coordenadas de su centro de gravedad y sus momentos de inercia. Estas tres cantidades por lo general se pueden encontrar en Softwares de modelado CAD. En el proyecto original estos se obtuvieron usando la API de FreeCAD en Python, aunque conociendo el funcionamiento de otras opciones, este procedimiento podría ser incluso innecesario.

## 2.2. Formato de Modelo MuJoCo
Para definir las características del entorno de modelado, MuJoCo utiliza el formato nativo MJCF, un lenguaje basado en XML. En este archivo el usuario puede definir:
- **Uniones entre elementos**
- **Sensores y Actuadores**
- **Parámetros físicos de la simulación**: Gravedad, diferencial de tiempo
- **Visualización y Renderizado**

MuJoCo ofrece una amplia documentación respecto a cómo configurar cada uno de estos parámetros. A pesar de ello, aquí se presenta una breve introducción. 

**USO DE IA:** El esqueleto del archivo MJCF en este archivo se generó usando Gemini. Este archivo se generó al consultarle a la IA por una serie de pasos para construir el formato. Los pasos se resumen a continuación (escritos a mano). 

Observe en el archivo balancebot.xml el esqueleto del MJCF define primero parámetros del compilador (ángulo en radianes, la carpeta donde encontrar las mallas y el diferencial de tiempo máximo). Luego se configura la opción del diferencial de tiempo máximo.

La importación de archivos STL a MuJoCo se realiza dentro del bloque asset. Cada malla se importa a través de una línea <mesh ...>, donde se escribe el nombre de los ficheros y su escala. Observe que se aplica un factor de escalamiento en los tres ejes de 0.001 dado que el estandar internacional de dimensión en softwares de diseño es el milímetro. Al dividir entre 1000 se obtienen dimensiones en metros.

En worldbody se definen ambas las propiedades del entorno como del robot mismo. Observe que en geom se define la geometría del piso, donde se incluye su tamaño y su coeficiente de fricción.

El cuerpo del robot se define tramién dentro de worldbody. Observe que las *masas* del robot se representan a través <inertial> y las geometrías para colisiones son geom. Cualquier ubicación relevante, que no corresponda a una malla o un objeto con masa, puede definirse a través de una línea site. En el robot definido, hay un único site que corresponde a la ubicación física del sensor IMU, Las ruedas se acoplan al chasis del robot a través de juntas (joints). Al igual que en los eslabones de un mecanismo el tipo de junta dicta cuántos grados de libertad tiene y qué tipos de movimiento puede realizar (traslacional, rotacional o complejo). Una junta de tipo 'hinge' corresponde a una junta de bisagra, un tipo de junta que solo puede rotar alrededor de un eje.

Finalmente se define un bloque actuator, donde se definen los motores de cada rueda. Observe que entre los parametros de <motor> se incluye la razón de la caja reductora, que representa la reducción de velocidad del eje del motor al eje de salida. También se fefine el rango de control del motor.

En el bloque sensor se agregan el acelerómetro y el giroscopio en el site definido anteriormente. En este bloque también se agregan la posición y velocidad de los motores con encoders. Sin embargo, al igual que como se indica en los videos, estos últimos cuatro valores sólo existen dentro de la simulación.

## 1.3. Aspectos de Implementación que pueden cambiar
Dado que este proyecto está intencionado para poder implementar un agente con cuerpo (embodied agent), es importante hacer una observación acerca de cómo el robot puede medir el estado de movimiento de sus motores.

Para medir la posición y velocidad angular de un motor eléctrico, el método utilizado por el BALA5 de Digikey es emplear *encoders*: sensores optomecánicos con ruedas ranuradas que codifican la posición angular de un eje con estados de luz y oscuridad.

No obstante, si queremos ser más tacaños (voy a quitar esto más adelante), vale la pena hacer un pequeño cambio sobre esta parte. En un robot más simple el control de los motores se podría realizar a través de una técnica conocida como PWM. En esta el voltaje entregado a los motores se produce de forma pulsada, con un ancho de pulso que varía entre 0 hasta el periodo completo. Esto permite controlar la velocidad del motor, aunque es necesario de todas maneras un encoder para determinar el estado de los motores....





# 3. ENTORNO DE GYMNASIUM


In [2]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import mujoco

In [4]:
class BalanceBotEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 50}

    def __init__(self, mjcf_path):
        super().__init__()
        
        # Cargar modelo y datos de MuJoCo
        self.model = mujoco.MjModel.from_xml_path(mjcf_path)
        self.data = mujoco.MjData(self.model)
        
        # Definir espacio de acciones (torque motores izq y der, normalizado de -1 a 1)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)
        
        # Definir espacio de observaciones (ej. pitch, pitch_rate, vel_wheel_l, vel_wheel_r)
        high_obs = np.array([np.pi, 20.0, 50.0, 50.0], dtype=np.float32)
        self.observation_space = spaces.Box(low=-high_obs, high=high_obs, dtype=np.float32)
        
        self.max_tilt = np.deg2rad(30) # 30 grados de límite antes de caer

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        
        # Opcional: Aplicar perturbación inicial aleatoria para el entrenamiento
        self.data.qpos[0] = 0.0 # Posición inicial
        
        observation = self._get_observation()
        info = {}
        return observation, info

    def step(self, action):
        # 1. Aplicar la acción a los actuadores
        self.data.ctrl[:] = action
        
        # 2. Avanzar un paso en la simulación de MuJoCo
        mujoco.mj_step(self.model, self.data)
        
        # 3. Obtener nuevas observaciones
        observation = self._get_observation()
        
        # 4. Calcular recompensa y comprobar si el robot cayó (terminated)
        pitch = observation[0]
        terminated = bool(abs(pitch) > self.max_tilt)
        truncated = False
        
        reward = 1.0 if not terminated else 0.0 # Ejemplo básico de recompensa
        info = {}
        
        return observation, reward, terminated, truncated, info

    def _get_observation(self):
        # Extraer datos de sensores / cinemática para armar el vector de observación
        # (Aquí implementarías la lectura de tu IMU y encoders según tu archivo XML)
        pitch = self.data.qpos[1] if self.model.nq > 1 else 0.0
        pitch_rate = self.data.qvel[1] if self.model.nv > 1 else 0.0
        wheel_vel_l = self.data.qvel[2] if self.model.nv > 2 else 0.0
        wheel_vel_r = self.data.qvel[3] if self.model.nv > 3 else 0.0
        
        return np.array([pitch, pitch_rate, wheel_vel_l, wheel_vel_r], dtype=np.float32)

    def render(self):
        # Implementar lógica de renderizado si se usa visor interactivo
        pass

    def close(self):
        pass

In [6]:
from gymnasium.utils.env_checker import check_env

env = BalanceBotEnv(mjcf_path="autobalance.xml") # Ajusta la ruta a tu archivo XML
check_env(env)
print("¡El entorno de Gymnasium cumple con los estándares correctamente!")

¡El entorno de Gymnasium cumple con los estándares correctamente!


c:\Users\mario\AppData\Local\Programs\Python\Python310\lib\site-packages\gymnasium\utils\env_checker.py:440: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantiating the environment through `gymnasium.make`
  logger.warn(


In [ ]:
'''
# DESCOMENTAR PARA VER SIMULADOR

import mujoco
import mujoco.viewer
import time

# Cargar el modelo y los datos utilizando tu archivo XML (MJCF)
model = mujoco.MjModel.from_xml_path("autobalance.xml")
data = mujoco.MjData(model)

# Lanzar el visor pasivo interactivo en una ventana independiente
with mujoco.viewer.launch_passive(model, data) as viewer:
    # Mantener la simulación corriendo durante un número determinado de pasos
    while viewer.is_running():
        # Avanzar un paso en la simulación física (timestep configurado en el XML)
        mujoco.mj_step(model, data)
        
        # Sincronizar el visor gráfico con los nuevos datos de la simulación
        viewer.sync()
        
        # Opcional: Agregar una pequeña pausa para que la visualización no vaya demasiado rápido
        time.sleep(0.005)
'''

KeyboardInterrupt: 

: 